# C군(신규형 비금융) + Bureau 전처리 노트북 (v2)

**목적**: 기존 `model_dataset.csv`(55컬럼, A+B군+EXT_SOURCE)에 아래를 추가 병합한다.

| 추가 그룹 | 내용 | 개수 |
|---|---|---|
| C군 (비금융) | 사회연결망 2 + FLAG_CONT_MOBILE + 인구통계 3 + FLAG_OWN_CAR | 7 |
| A_financial 추가 | AMT_REQ_CREDIT_BUREAU_* | 6 |
| A_financial 추가 | bureau.csv 집계변수 | 6 |

(installments_payments.csv는 이번 버전에서 제외)

컬럼명은 실제 업로드된 `application_train.csv`, `bureau.csv`, `model_dataset.csv`로
**직접 확인 완료**한 것들이라 그대로 실행 가능함 (SK_ID_CURR 100% 매칭 확인됨).

병합 순서: `model_dataset.csv`(55) → +C군/AMT_REQ(application_train) → +bureau 집계(6)
= 최종 약 **61 + 15 = 76~79컬럼** (원핫 인코딩 개수에 따라 유동적, 실행 후 정확한 수 확인)


## 0. 환경 설정 및 파일 경로

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np

# ---- 아래 경로를 본인 환경에 맞게 수정 ----
BASE_DIR = '/content/drive/MyDrive/BOOSTMAP/데이터/원본'

PATH_MODEL_DATASET   = f'{BASE_DIR}/model_dataset.csv'          # 기존 55컬럼 파일
PATH_APPLICATION     = f'{BASE_DIR}/application_train.csv'      # Kaggle 원본 (train, TARGET 포함)
PATH_BUREAU          = f'{BASE_DIR}/bureau.csv'                 # Kaggle 원본

OUTPUT_PATH = f'{BASE_DIR}/model_dataset_v2.csv'


## 1. 기존 model_dataset.csv 로드

In [5]:
df_model = pd.read_csv(PATH_MODEL_DATASET)
print('model_dataset.csv shape:', df_model.shape)
assert 'SK_ID_CURR' in df_model.columns, 'SK_ID_CURR 키 컬럼이 없습니다. 병합 불가.'
df_model.head()


model_dataset.csv shape: (307511, 55)


,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_CONTRACT_TYPE_BIN,DTI,LTV_GOODS,CREDIT_TO_INCOME,...,ORGANIZATION_TYPE_G_Industry,ORGANIZATION_TYPE_G_Medicine,ORGANIZATION_TYPE_G_Other,ORGANIZATION_TYPE_G_Self-employed,ORGANIZATION_TYPE_G_Trade,NAME_INCOME_TYPE_G_Commercial associate,NAME_INCOME_TYPE_G_Other,NAME_INCOME_TYPE_G_Pensioner,NAME_INCOME_TYPE_G_State servant,NAME_INCOME_TYPE_G_Working
0,100002,1,202500.0,406597.5,24700.5,351000.0,0,0.121978,1.158397,2.007889,...,0,0,0,0,0,0,0,0,0,1
1,100003,0,270000.0,1293502.5,35698.5,1129500.0,0,0.132217,1.145199,4.790750,...,0,0,1,0,0,0,0,0,1,0
2,100004,0,67500.0,135000.0,6750.0,135000.0,1,0.100000,1.000000,2.000000,...,0,0,0,0,0,0,0,0,0,1
3,100006,0,135000.0,312682.5,29686.5,297000.0,0,0.219900,1.052803,2.316167,...,0,0,0,0,0,0,0,0,0,1
4,100007,0,121500.0,513000.0,21865.5,513000.0,0,0.179963,1.000000,4.222222,...,0,0,1,0,0,0,0,0,0,1


## 2. C군(비금융) 원본 컬럼 + AMT_REQ_CREDIT_BUREAU_* 가져오기

⚠️ **반드시 `application_train.csv`를 써야 함** (TARGET 있는 학습용 원본).
`application_test.csv`를 쓰면 SK_ID_CURR이 model_dataset과 전혀 겹치지 않아 전부 결측 처리됨
(실제로 이 문제를 한 번 겪었으니 재확인).


In [6]:
C_GROUP_COLS = [
    'SK_ID_CURR',
    # C군 (비금융, 7개)
    'OBS_30_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE',
    'FLAG_CONT_MOBILE',
    'CNT_CHILDREN',
    'NAME_EDUCATION_TYPE',
    'NAME_HOUSING_TYPE',
    'FLAG_OWN_CAR',
    # A_financial 추가 (신용조회 횟수, 6개)
    'AMT_REQ_CREDIT_BUREAU_HOUR',
    'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR',
]

df_app = pd.read_csv(PATH_APPLICATION, usecols=lambda c: c in C_GROUP_COLS)
print('application_train.csv (C군+AMT_REQ) shape:', df_app.shape)
missing_cols = set(C_GROUP_COLS) - set(df_app.columns)
if missing_cols:
    print('⚠️ 원본에 없는 컬럼 (컬럼명 재확인 필요):', missing_cols)

# model_dataset과 SK_ID_CURR 매칭 여부 사전 확인 (test 파일 오사용 방지)
overlap_ratio = len(set(df_app['SK_ID_CURR']) & set(df_model['SK_ID_CURR'])) / len(df_model)
print(f'model_dataset과 SK_ID_CURR 매칭률: {overlap_ratio:.2%}')
assert overlap_ratio > 0.95, '⚠️ 매칭률이 너무 낮습니다. application_train.csv가 맞는지 확인하세요 (test 파일 아닌지).'

df_app.head()


application_train.csv (C군+AMT_REQ) shape: (307511, 14)
model_dataset과 SK_ID_CURR 매칭률: 100.00%


,SK_ID_CURR,FLAG_OWN_CAR,CNT_CHILDREN,NAME_EDUCATION_TYPE,NAME_HOUSING_TYPE,FLAG_CONT_MOBILE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,N,0,Secondary / secondary special,House / apartment,1,2.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,N,0,Higher education,House / apartment,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,Y,0,Secondary / secondary special,House / apartment,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,N,0,Secondary / secondary special,House / apartment,1,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,N,0,Secondary / secondary special,House / apartment,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# AMT_REQ_CREDIT_BUREAU_* 결측치: '조회 이력 없음'으로 보고 0 처리 + 원래 결측 플래그 보존
req_cols = [c for c in df_app.columns if c.startswith('AMT_REQ_CREDIT_BUREAU')]
df_app['AMT_REQ_CREDIT_BUREAU_MISSING_FLAG'] = df_app[req_cols].isna().any(axis=1).astype(int)
df_app[req_cols] = df_app[req_cols].fillna(0)

# FLAG_OWN_CAR: 'Y'/'N' -> 1/0
df_app['FLAG_OWN_CAR'] = df_app['FLAG_OWN_CAR'].map({'Y': 1, 'N': 0})

# 사회연결망 변수 결측치: 없음=0으로 처리 + 플래그
social_cols = ['OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE']
df_app['SOCIAL_CIRCLE_MISSING_FLAG'] = df_app[social_cols].isna().any(axis=1).astype(int)
df_app[social_cols] = df_app[social_cols].fillna(0)

print(df_app.isna().sum())


SK_ID_CURR                            0
FLAG_OWN_CAR                          0
CNT_CHILDREN                          0
NAME_EDUCATION_TYPE                   0
NAME_HOUSING_TYPE                     0
FLAG_CONT_MOBILE                      0
OBS_30_CNT_SOCIAL_CIRCLE              0
DEF_30_CNT_SOCIAL_CIRCLE              0
AMT_REQ_CREDIT_BUREAU_HOUR            0
AMT_REQ_CREDIT_BUREAU_DAY             0
AMT_REQ_CREDIT_BUREAU_WEEK            0
AMT_REQ_CREDIT_BUREAU_MON             0
AMT_REQ_CREDIT_BUREAU_QRT             0
AMT_REQ_CREDIT_BUREAU_YEAR            0
AMT_REQ_CREDIT_BUREAU_MISSING_FLAG    0
SOCIAL_CIRCLE_MISSING_FLAG            0
dtype: int64


**NAME_EDUCATION_TYPE / NAME_HOUSING_TYPE 원-핫 인코딩** (`_C` 접미사 규칙)

In [8]:
for col in ['NAME_EDUCATION_TYPE', 'NAME_HOUSING_TYPE']:
    df_app[col] = df_app[col].fillna('Unknown')

df_app_encoded = pd.get_dummies(
    df_app,
    columns=['NAME_EDUCATION_TYPE', 'NAME_HOUSING_TYPE'],
    prefix=['NAME_EDUCATION_TYPE_C', 'NAME_HOUSING_TYPE_C'],
    dummy_na=False
)

print('원핫 인코딩 후 shape:', df_app_encoded.shape)

# 이전에 겪었던 원핫 인코딩 누락 버그(NAME_INCOME_TYPE_G, 41% 미포함) 재발 방지 검증
edu_dummy_sum = df_app_encoded.filter(like='NAME_EDUCATION_TYPE_C').sum(axis=1)
house_dummy_sum = df_app_encoded.filter(like='NAME_HOUSING_TYPE_C').sum(axis=1)
assert (edu_dummy_sum == 1).all(), '⚠️ NAME_EDUCATION_TYPE 원핫 인코딩 누락 발견'
assert (house_dummy_sum == 1).all(), '⚠️ NAME_HOUSING_TYPE 원핫 인코딩 누락 발견'
print('원핫 인코딩 검증 통과')


원핫 인코딩 후 shape: (307511, 25)
원핫 인코딩 검증 통과


## 3. bureau.csv 집계변수 (6개) — Character(상환의지) 축 보강

1. `BUREAU_LOAN_COUNT` — 과거 대출건수(타 기관 전체)
2. `BUREAU_ACTIVE_LOAN_COUNT` — 현재 활성 대출수
3. `BUREAU_DAYS_OVERDUE_MAX` — 연체일수 최대
4. `BUREAU_DAYS_OVERDUE_MEAN` — 연체일수 평균
5. `BUREAU_DEBT_RATIO` — 부채비율 (잔여부채합 / 총여신합)
6. `BUREAU_NO_HISTORY_FLAG` — 타 기관 대출 이력 자체가 없음 (씬파일러 플래그)


In [9]:
df_bureau = pd.read_csv(PATH_BUREAU)
print('bureau.csv shape:', df_bureau.shape)
df_bureau.head()


bureau.csv shape: (1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [10]:
bureau_agg = df_bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_LOAN_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_DAYS_OVERDUE_MAX=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_DAYS_OVERDUE_MEAN=('CREDIT_DAY_OVERDUE', 'mean'),
    _SUM_CREDIT_SUM=('AMT_CREDIT_SUM', 'sum'),
    _SUM_CREDIT_SUM_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
).reset_index()

bureau_agg['BUREAU_DEBT_RATIO'] = np.where(
    bureau_agg['_SUM_CREDIT_SUM'] > 0,
    bureau_agg['_SUM_CREDIT_SUM_DEBT'] / bureau_agg['_SUM_CREDIT_SUM'],
    0
)
bureau_agg = bureau_agg.drop(columns=['_SUM_CREDIT_SUM', '_SUM_CREDIT_SUM_DEBT'])

print('bureau 집계 shape (SK_ID_CURR 단위):', bureau_agg.shape)
bureau_agg.head()


bureau 집계 shape (SK_ID_CURR 단위): (305811, 6)


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_LOAN_COUNT,BUREAU_DAYS_OVERDUE_MAX,BUREAU_DAYS_OVERDUE_MEAN,BUREAU_DEBT_RATIO
0,100001,7,3,0,0.0,0.410555
1,100002,8,2,0,0.0,0.284122
2,100003,4,1,0,0.0,0.000000
3,100004,2,0,0,0.0,0.000000
4,100005,3,2,0,0.0,0.864992


In [11]:
all_ids = df_model[['SK_ID_CURR']].drop_duplicates()
bureau_agg_full = all_ids.merge(bureau_agg, on='SK_ID_CURR', how='left')
bureau_agg_full['BUREAU_NO_HISTORY_FLAG'] = bureau_agg_full['BUREAU_LOAN_COUNT'].isna().astype(int)

fill_cols = ['BUREAU_LOAN_COUNT', 'BUREAU_ACTIVE_LOAN_COUNT',
             'BUREAU_DAYS_OVERDUE_MAX', 'BUREAU_DAYS_OVERDUE_MEAN', 'BUREAU_DEBT_RATIO']
bureau_agg_full[fill_cols] = bureau_agg_full[fill_cols].fillna(0)

print('이력 없음(씬파일러) 비율: {:.2%}'.format(bureau_agg_full['BUREAU_NO_HISTORY_FLAG'].mean()))
bureau_agg_full.head()


이력 없음(씬파일러) 비율: 14.31%


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_LOAN_COUNT,BUREAU_DAYS_OVERDUE_MAX,BUREAU_DAYS_OVERDUE_MEAN,BUREAU_DEBT_RATIO,BUREAU_NO_HISTORY_FLAG
0,100002,8.0,2.0,0.0,0.0,0.284122,0
1,100003,4.0,1.0,0.0,0.0,0.000000,0
2,100004,2.0,0.0,0.0,0.0,0.000000,0
3,100006,0.0,0.0,0.0,0.0,0.000000,1
4,100007,1.0,0.0,0.0,0.0,0.000000,0


## 4. 전체 병합 → model_dataset_v2.csv

In [12]:
df_final = df_model.merge(df_app_encoded, on='SK_ID_CURR', how='left')
df_final = df_final.merge(bureau_agg_full, on='SK_ID_CURR', how='left')

print('최종 병합 shape:', df_final.shape)
print('원본 model_dataset 행 수와 동일한지 확인:', len(df_final) == len(df_model))


최종 병합 shape: (307511, 85)
원본 model_dataset 행 수와 동일한지 확인: True


In [13]:
# 최종 결측치 점검
na_summary = df_final.isna().sum()
na_summary = na_summary[na_summary > 0]
if len(na_summary) > 0:
    print('⚠️ 결측치가 남아있는 컬럼 (재확인 필요):')
    print(na_summary)
else:
    print('결측치 없음 - 병합 정상')


결측치 없음 - 병합 정상


In [14]:
print('=== 최종 컬럼 목록 ===')
print(f'총 {df_final.shape[1]}개 컬럼, {df_final.shape[0]}행')
for c in df_final.columns:
    print(' -', c)


=== 최종 컬럼 목록 ===
총 85개 컬럼, 307511행
 - SK_ID_CURR
 - TARGET
 - AMT_INCOME_TOTAL
 - AMT_CREDIT
 - AMT_ANNUITY
 - AMT_GOODS_PRICE
 - NAME_CONTRACT_TYPE_BIN
 - DTI
 - LTV_GOODS
 - CREDIT_TO_INCOME
 - ANNUITY_TO_CREDIT
 - AMT_CREDIT_LOG
 - AMT_ANNUITY_LOG
 - AMT_GOODS_PRICE_LOG
 - AMT_INCOME_TOTAL_LOG
 - AMT_INCOME_OUTLIER
 - EXT_SOURCE_1
 - EXT_SOURCE_2
 - EXT_SOURCE_3
 - EXT_SOURCE_1_MISSING
 - EXT_SOURCE_2_MISSING
 - EXT_SOURCE_3_MISSING
 - AGE
 - YEARS_EMPLOYED
 - DAYS_EMPLOYED_ANOM
 - FLAG_OWN_REALTY_BIN
 - OCCUPATION_TYPE_C_Cleaning staff
 - OCCUPATION_TYPE_C_Cooking staff
 - OCCUPATION_TYPE_C_Core staff
 - OCCUPATION_TYPE_C_Drivers
 - OCCUPATION_TYPE_C_HR staff
 - OCCUPATION_TYPE_C_High skill tech staff
 - OCCUPATION_TYPE_C_IT staff
 - OCCUPATION_TYPE_C_Laborers
 - OCCUPATION_TYPE_C_Low-skill Laborers
 - OCCUPATION_TYPE_C_Managers
 - OCCUPATION_TYPE_C_Medicine staff
 - OCCUPATION_TYPE_C_Private service staff
 - OCCUPATION_TYPE_C_Realty agents
 - OCCUPATION_TYPE_C_Sales staff
 - OCCUP

In [15]:
df_final.to_csv(OUTPUT_PATH, index=False)
print(f'저장 완료: {OUTPUT_PATH}')


저장 완료: /content/drive/MyDrive/BOOSTMAP/데이터/원본/model_dataset_v2.csv


## 5. 다음 단계 체크리스트

- [ ] 위 컬럼 수를 팀 문서(`변수선정.csv`)의 예상치와 대조하여 갱신
- [ ] `변수선정.csv`에 C군 7개 + bureau 파생 6개 + AMT_REQ 6개(+플래그 2개) 신규 행 추가
- [ ] IV / 그룹기여도(leave-one-out) / EXT상관 **재실행**
- [ ] 씬파일러(`BUREAU_NO_HISTORY_FLAG`=1, 약 14.3%) 표본 분리 여부 팀 논의 재개
- [ ] bureau 반영 후 M1(금융) 변수 구성 갱신 필요
- [ ] installments_payments.csv는 이번 버전 미포함 — 추후 필요시 별도 추가
